In [36]:
import random
import pandas as pd
from collections import defaultdict
import itertools

In [40]:
# This function evaluates EV of each action independently
def blackjack_basic_strategy_MC(
        number_of_decks = 6,
        sims = 50000,
        deck_pen = 0.75,
        dealer_hit_soft_17 = True,
        double_allowed = True,
        split_allowed = False, #not included
        surrender_allowed = False, #not included
        insurance_allowed = False, #not included
        blackjack_payout = 1.5
):
    # Card Setup -------------------------------------------------
        bj_rank = ['A','2','3','4','5','6','7','8','9','10','10','10','10']
        single_deck = bj_rank * 4
        game_decks = single_deck * number_of_decks

        # All possible staring hands 
        bj_rank_unique = ['A','2','3','4','5','6','7','8','9','10']
        player_cards = []
        for i in bj_rank_unique:
                for j in bj_rank_unique:
                        if j < i:
                                continue
                        else:
                                player_cards.append([i,j])

        dealer_face = bj_rank_unique.copy()

    # Result Storage -------------------------------------------------
        allowed_move = ['stand', 'hit']

        if double_allowed == True:
                allowed_move.append('double')
        if split_allowed == True:
                allowed_move.append('split')
        if surrender_allowed == True:
                allowed_move.append('surrender')
        if insurance_allowed == True:
                allowed_move.append('insurance')

        results = defaultdict(lambda:{'winning': 0, 'stake': 0, 'count':0})

        def record(dealer_up, player_initial, move, profit, stake):
                key = (dealer_up, ','.join(sorted(player_initial)), move)
                results[key]['winning'] += profit
                results[key]['stake'] += stake
                results[key]['count'] += 1

    # Hand Value -------------------------------------------------
        def hand_values(card_list):
                output = 0
                aces = 0
                for i in card_list:
                        if i == 'A':
                                aces += 1
                                output += 11
                        else:
                                output += int(i)
                while output > 21 and aces > 0:
                        output -= 10
                        aces -= 1
                return output    
        
        def is_soft(hand):
                return 'A' in hand and hand_values(hand) <= 21 and hand_values(hand) - 10 >= 12

    # Dealer Move -------------------------------------------------
        def dealer_move(dealer_cards, deck_slice):
                cards = dealer_cards.copy()
                i = 0
                while True:
                        value = hand_values(cards)
                        if value > 21:
                                return cards, True, i # dealer bust
                        if value >= 17 and (value > 17 or not (dealer_hit_soft_17 and is_soft(cards))):  
                                return cards, False, i
                        if i >= len(deck_slice):
                                return cards, value > 21, i
                        cards.append(deck_slice[i])
                        i += 1       

    # Player Moves -------------------------------------------------
        def stand(player_cards, dealer_cards, deck_slice):
                dealer_fin, dealer_busted, num_dealer_cards = dealer_move(dealer_cards, deck_slice)
                player_value = hand_values(player_cards)
                dealer_value = hand_values(dealer_fin)
                if dealer_busted or player_value > dealer_value:
                        record(dealer_cards[0], player_cards[:2], 'stand', 1, 1)
                elif player_value == dealer_value: #push
                        record(dealer_cards[0], player_cards[:2], 'stand', 0, 1)
                else:
                        record(dealer_cards[0], player_cards[:2], 'stand', -1, 1)
                
        def hit(player_cards, dealer_cards, deck_slice):
                cards = player_cards.copy()
                cards.append(deck_slice[0])
                player_value = hand_values(cards)
                if player_value > 21:
                        record(dealer_cards[0], player_cards[:2], 'hit', -1, 1)
                else:
                        dealer_fin, dealer_busted, num_dealer_cards = dealer_move(dealer_cards, deck_slice[1:]) 
                        dealer_value = hand_values(dealer_fin)
                        if dealer_busted or player_value > dealer_value:
                                record(dealer_cards[0], player_cards[:2], 'hit', 1, 1)
                        elif player_value == dealer_value: #push
                                record(dealer_cards[0], player_cards[:2], 'hit', 0, 1)
                        else:
                                record(dealer_cards[0], player_cards[:2], 'hit', -1, 1)                       
       
        def double(player_cards, dealer_cards, deck_slice):
                cards = player_cards.copy()
                cards.append(deck_slice[0])
                player_value = hand_values(cards)
                if player_value > 21:
                        record(dealer_cards[0], player_cards[:2], 'double', -2, 2)
                else:
                        dealer_fin, dealer_busted, num_dealer_cards = dealer_move(dealer_cards, deck_slice[1:]) 
                        dealer_value = hand_values(dealer_fin)
                        if dealer_busted or player_value > dealer_value:
                                record(dealer_cards[0], player_cards[:2], 'double', 2, 2)
                        elif player_value == dealer_value: #push
                                record(dealer_cards[0], player_cards[:2], 'double', 0, 2)
                        else:
                                record(dealer_cards[0], player_cards[:2], 'double', -2, 2)                
        
        def split(player_cards, dealer_cards, deck_slice):
                return
        def surrender(player_cards, dealer_cards, deck_slice):
                return
        def insurance(player_cards, dealer_cards, deck_slice):
                return

    # Sim -------------------------------------------------

        total_cards = len(game_decks)
        pen_limit = int(total_cards * deck_pen)
        games = 0

        while games < sims:
                current_deck = game_decks.copy()
                random.shuffle(current_deck)
                pos = 0

                while pos < pen_limit:
                        player_initial = [current_deck[pos], current_deck[pos+1]]
                        dealer_cards = [current_deck[pos+2], current_deck[pos+3]]
                        pos += 4
                        deck_slice = current_deck[pos:]

                        #check for blackjack first 
                        if hand_values(player_initial) == 21:
                                if hand_values(dealer_cards) == 21:
                                        record(dealer_cards[0], player_initial[:2], 'blackjack', 0, 1) # push
                                else:
                                        record(dealer_cards[0], player_initial[:2], 'blackjack', 1 * blackjack_payout, 1) # push
                        else:
                                stand(player_initial, dealer_cards, deck_slice)
                                dealer_fin, dealer_busted, num_dealer_cards = dealer_move(dealer_cards, deck_slice[1:]) # untidy, this does not need to be in every function - remove if have time
                                hit(player_initial, dealer_cards, deck_slice)
                                if 'double' in allowed_move:
                                        double(player_initial, dealer_cards, deck_slice)
                                if 'split' in allowed_move:
                                        split(player_initial, dealer_cards, deck_slice)
                                if 'surrender' in allowed_move:
                                        surrender(player_initial, dealer_cards, deck_slice)
                                if 'insurance' in allowed_move:
                                        insurance(player_initial, dealer_cards, deck_slice)                      
                        pos += 1 + num_dealer_cards # for player move and dealer cards         

                games += 1

    # Results Table -------------------------------------------------
        
        results_df = pd.DataFrame(results).T.reset_index()
        results_df.columns = ['dealer_up', 'player_hand', 'move', 'winning', 'stake', 'count']
        results_df['EV'] = results_df['winning'] / results_df['stake']

        ev_table = results_df.pivot_table(
        index=['dealer_up', 'player_hand'],
        columns='move',
        values='EV'
        )

        ev_table['avg_EV'] = ev_table.mean(axis=1)
        ev_table = ev_table.sort_values(by='avg_EV', ascending=True)
        ev_table = ev_table.drop(columns='avg_EV')

        ev_table = ev_table.round(4)
        ev_table = ev_table.sort_index()
        

        return ev_table 



In [41]:
test = blackjack_basic_strategy_MC(
        number_of_decks = 1,
        sims = 50000,
        deck_pen = 0.75,
        dealer_hit_soft_17 = True,
        double_allowed = True,
        split_allowed = False, #not included
        surrender_allowed = False, #not included
        insurance_allowed = False, #not included
        blackjack_payout = 1.5
)

In [42]:
test

move                   blackjack  double     hit   stand
dealer_up player_hand                                   
10        10,10              NaN -0.8429 -0.8429  0.4426
          10,2               NaN -0.4288 -0.4288 -0.5744
          10,3               NaN -0.4158 -0.4158 -0.5889
          10,4               NaN -0.4920 -0.4920 -0.5766
          10,5               NaN -0.5219 -0.5219 -0.5677
...                          ...     ...     ...     ...
A         8,9                NaN -0.7134 -0.7134 -0.6710
          8,A                NaN -0.3876 -0.3876 -0.1628
          9,9                NaN -0.7778 -0.7778 -0.3333
          9,A                NaN -0.3551 -0.3551  0.0471
          A,A                NaN -0.4386 -0.4386 -0.6491

[550 rows x 4 columns]

In [ ]:
def blackjack_basic_strategy_deterministic(number_of_decks = 1,
             sims = 10000,
             deck_pen = 0.3,
             dealer_hit_soft_17 = True,
             double_allowed = True,
             split_allowed = True,
             surrender_allowed = True,
             insurance_allowed = True,
             blackjack_payout = 1.5
             ):

    
    return 

In [ ]:
def blackjack_basic_strategy_MC(number_of_decks = 1,
             sims = 10000,
             deck_pen = 0.3,
             dealer_hit_soft_17 = True,
             double_allowed = True,
             split_allowed = True,
             surrender_allowed = True,
             insurance_allowed = True,
             blackjack_payout = 1.5
             ):
    # card setup ------------------------------------------------
    card_values = {
        #'A' : [1, 11], note used
        '2' : [2],
        '3' : [3],
        '4' : [4],
        '5' : [5],
        '6' : [6],
        '7' : [7],
        '8' : [8],
        '9' : [9],
        '10' : [10],
        'J' : [10],
        'Q' : [10],
        'K' : [10]
    }
        
    # rank = ['A','2','3','4','5','6','7','8','9','10','J','Q','K']
    # as suit not important and face cards just have a value of 10:
    bj_rank = ['A','2','3','4','5','6','7','8','9','10','10','10','10']
    single_deck = bj_rank * 4
    game_decks = single_deck * number_of_decks
    
    bj_rank_unique = ['A','2','3','4','5','6','7','8','9','10']
    player_cards = []
    for i in bj_rank_unique:
        for j in bj_rank_unique:
            if j < i:
                continue
            else:
                player_cards.append(i,j)

    dealer_face = bj_rank_unique.copy()

    # setting up results table ------------------------------------------------
    allowed_move = ['stand', 'hit']

    if double_allowed == True:
        allowed_move.append('double')
    if split_allowed == True:
        allowed_move.append('split')
    if surrender_allowed == True:
        allowed_move.append('surrender')
    if insurance_allowed == True:
        allowed_move.append('insurance')

    results = {}

    for dealer in dealer_face:
        for hand in player_cards:
            hand_key = ','.join(sorted(hand))
            for move in allowed_move:
                key = (dealer, hand_key, move)
                results[key] = {"winning": 0, "stake": 0}

    def update_results(player_cards, dealer_cards, move, stake):
        player_hand_value = hand_values(player_cards)
        dealer_hand_value = hand_values(dealer_cards)
        if dealer_hand_value > 21: # dealer bust
            results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), move)]["winning"] += stake * 2
            results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), move)]["stake"] += stake
        elif player_hand_value == dealer_hand_value: # push
            results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), move)]["winning"] += stake 
            results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), move)]["stake"] += stake
        elif player_hand_value > 21 or player_hand_value < dealer_hand_value: #player lost
            results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), move)]["stake"] += stake
        else: #player won
            results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), move)]["winning"] += stake * 2
            results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), move)]["stake"] += stake
    # ------------------------------------------------
    # Player Moves

    #### need to update so that a copy of the player and dealer hand as well as the current count is taken otherwise we override it

    def deal_cards(current_card):
        player_cards = [game_decks[current_card], game_decks[current_card + 2]]
        dealer_cards = [game_decks[current_card + 1], game_decks[current_card + 3]]
        current_card = current_card + 4
        return player_cards, dealer_cards, current_card
    
    def hit(pc_hit, dc_hit, cc_hit):

        pc_hit.append(game_decks[cc_hit])
        cc_hit += 1 
        if hand_values(pc_hit) < 22: #player not busted
            dc, cc_hit = dealer_move(dc_hit, cc_hit)
        update_results(pc_hit, dc_hit, 'hit', 1)
        return cc_hit
    
    def stand(pc_stand, dc_stand, cc_stand):
        dc_stand, cc_stand = dealer_move(dc_stand, cc_stand)
        update_results(pc_stand, dc_stand, 'stand', 1)
        return cc_stand
    
    def double(pc_double, dc_double, cc_double):
        pc_double.append(game_decks[cc_double])
        cc_double += 1 
        if hand_values(pc_double) < 22: #player not busted
            dc, cc = dealer_move(dc_double, cc_double)
        update_results(pc_double, dc_double, 'double', 2)
        return cc_double
    #will do later
    #def split():
    #    return
    #def surrender():
    #    return 
    #def insurance():
    #    return
    
    def play_legal_moves(player_cards, dealer_cards, current_card):
        # player Blackjack
        if hand_values(player_cards) == 21:
            #Dealer also has blackjack
            if hand_values(dealer_cards) == 21: 
                results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), 'stand')]["winning"] += 1  #push
                results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), 'stand')]["stake"] += 1
            else: 
                results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), 'stand')]["winning"] += 1 * (1 + blackjack_payout)
                results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), 'stand')]["stake"] += 1
            #dealer showing ace & blackjack
            if dealer_cards[0] == 'A' and hand_values(dealer_cards) == 21:
                results[(dealer_cards[0], ','.join(sorted(player_cards[:2])), 'stand')]["stake"] += 1
        else:
            pc_orig = player_cards.copy()
            dc_orig = dealer_cards.copy()
            cc_orig = current_card
            cc_out = current_card
            for i in allowed_move:
                if i == 'hit':
                    pc = pc_orig.copy()
                    dc = dc_orig.copy()
                    cc = cc_orig
                    print('hit')
                    print(pc)
                    print(dc)
                    print(cc)
                    cc_out = max(hit(pc, dc, cc), cc_out)

                if i == 'stand':
                    pc = pc_orig.copy()
                    dc = dc_orig.copy()
                    cc = cc_orig
                    print('stand')
                    print(pc)
                    print(dc)
                    print(cc)
                    cc_out = max(stand(pc, dc, cc), cc_out)

                if i == 'double':
                    pc = pc_orig.copy()
                    dc = dc_orig.copy()
                    cc = cc_orig
                    print('double')
                    print(pc)
                    print(dc)
                    print(cc)
                    cc_out = max(double(pc, dc, cc), cc_out)

        return cc_out
    
    # ------------------------------------------------
    # dealer move
    
    def dealer_move(dealer_cards, current_card):
        while True:
            value = hand_values(dealer_cards)
            soft = ('A' in dealer_cards) and value <= 17

            if value > 21:
                return dealer_cards, current_card
            if value > 17:  
                return dealer_cards, current_card
            if value == 17 and not dealer_hit_soft_17:
                return dealer_cards, current_card
            if value == 17 and dealer_hit_soft_17 and not soft:
                return dealer_cards, current_card

            # Hit
            dealer_cards.append(game_decks[current_card])
            current_card += 1 
        
    # ------------------------------------------------
    # hand value   
    def hand_values(card_list):
        output = 0
        aces = 0

        for i in card_list:
            if i == 'A':
                aces += 1
                output += 11
            else:
                output += card_values[i][0]

        while output > 21 and aces > 0:
            output -= 10
            aces -= 1
            
        return output    

    # ------------------------------------------------
    # simulation

    for i in range(sims):       
        current_card = 0
        
        random.shuffle(game_decks)
        while current_card < round(len(game_decks)*deck_pen):
            player_cards, dealer_cards, current_card = deal_cards(current_card)
            play_legal_moves(player_cards, dealer_cards, current_card)
    
    #creating an EV table
    results_df = pd.DataFrame(results).T
    results_df['ev'] = (results_df['winning']- results_df['stake']) / results_df['stake']

    return results_df

In [ ]:
#random.seed(1)

def game_sim(number_of_decks = 1,
             sims = 10000,
             deck_pen = 0.3,
             dealer_hit_soft_17 = True,
             double_allowed = True,
             split_allowed = True,
             surrender_allowed = True,
             insurance_allowed = True,
             blackjack_payout ='3:2'
             ):
    
    # ---------------------------------------------
    # setting up decks
    cards = ['A','2','3','4','5','6','7','8','9','10','J','Q','K']

    card_values = {
        #'A' : [1, 11], note used
        '2' : [2],
        '3' : [3],
        '4' : [4],
        '5' : [5],
        '6' : [6],
        '7' : [7],
        '8' : [8],
        '9' : [9],
        '10' : [10],
        'J' : [10],
        'Q' : [10],
        'K' : [10]
    }

    single_deck = cards * 4

    decks = single_deck * number_of_decks

    dealer_face = cards.copy() 

    player_cards = []
    for i in range(len(single_deck)):
        for j in range(len(single_deck)):
            if j < i:
                continue
            else:
                player_cards.append([single_deck[i],single_deck[j]])
    # ------------------------------------------------
    #setting up results table 

    allowed_move = ['stand', 'hit']

    if double_allowed == True:
        allowed_move.append('double')
    if split_allowed == True:
        allowed_move.append('split')
    if surrender_allowed == True:
        allowed_move.append('surrender')
    if insurance_allowed == True:
        allowed_move.append('insurance')

    results = []
    for i in dealer_face:
        for j in player_cards:
            for k in allowed_move:
                results.append({
                    'dealer_face': i, 
                    'player_cards': ','.join(sorted(j)),
                    'move': k,
                    'wins': 0,
                    'losses': 0,
                    'total': 0
                    })

    results_df = pd.DataFrame(results)

    # ------------------------------------------------
    #moves    
    def deal_cards(current_card):
        player_cards = [decks[current_card], decks[current_card + 2]]
        dealer_cards = [decks[current_card + 1], decks[current_card + 3]]
        current_card = current_card + 4
        return player_cards, dealer_cards, current_card
    
    def next_move(first_move, current_card, card_list):
        if first_move == 'Y':
            #add in other moves here
            # change this to moves later 
            first_move = random.choice(allowed_move)
            if first_move == 'hit':
                return hit(current_card, card_list, first_move) 
            if first_move == 'double':
                return double()
            if first_move == 'split':
                return split()
            if first_move == 'surrender':
                return surrender()
            if first_move == 'insurance':
                return insurance()
            else: 
                return stand(card_list, current_card, first_move)
        if hand_values(card_list) == 21:
            return stand(card_list, current_card, first_move)
        else:
            move = random.choice(['hit', 'stand'])
            if move == 'hit':
                return hit(current_card, card_list, first_move)
            else: 
                return stand(card_list, current_card, first_move)
    
    def hit(current_card, card_list, first_move):
        card_list.append(decks[current_card])
        current_card += 1
        if hand_values(card_list) > 21:
            return 'Bust', current_card, first_move
        else:
            return next_move(first_move, current_card, card_list)
    
    def stand(card_list, current_card, first_move):
        return hand_values(card_list), current_card, first_move
    
    # to do 
    def double():
        return 
    def split():
        return
    def surrender():
        return 
    def insurance():
        return
    
    def dealer_move(dealer_cards, current_card):

        while True:
            value = hand_values(dealer_cards)
            soft = ('A' in dealer_cards) and value -11 + 1 <= 10 

            if value > 21:
                return 'Bust', current_card
            if value > 17:  
                return value, current_card
            if value == 17 and not dealer_hit_soft_17:
                return value, current_card
            if value == 17 and dealer_hit_soft_17 and not soft:
                return value, current_card

            # Hit
            dealer_cards.append(decks[current_card])
            current_card += 1
         
    # this needs to deal with aces
    def hand_values(card_list):
        output = 0
        aces = 0

        for i in card_list:
            if i == 'A':
                aces += 1
                output += 11
            else:
                output += card_values[i][0]

        while output > 21 and aces > 0:
            output -= 10
            aces -= 1
            
        return output
    
    # ------------------------------------------------
    #running simulation
    for i in range(sims):
        
        #burn card - shouldn't have any impact
        current_card = 1
        decks = single_deck * number_of_decks
        random.shuffle(decks)
        while current_card < round(len(decks)*deck_pen):
            first_move = 'Y'
            player_cards, dealer_cards, current_card = deal_cards(current_card)

            #blackjack
            if hand_values(player_cards) == 21:
                if hand_values(dealer_cards) == 21: 
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == 'stand'), 'total' ] += 1
                else:
                    if blackjack_payout == '3:2':
                        multiplier = 1.5
                    else:
                        multiplier = 1.2
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == 'stand'), 'wins' ] += 1 * multiplier
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == 'stand'), 'total' ] += 1
            
            #dealer showing ace & blackjack
            if dealer_cards[0] == 'A' and hand_values(dealer_cards) == 21:
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == 'stand'), 'losses' ] += 1
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == 'stand'), 'total' ] += 1

            
            player_result, current_card, first_move = next_move(first_move, current_card, player_cards.copy())                  

            if player_result == 'Bust':
                results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == first_move), 'losses' ] += 1
                results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == first_move), 'total' ] += 1
            else:
                dealer_result, current_card = dealer_move(dealer_cards.copy(), current_card)
                if dealer_result == 'Bust':
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == first_move), 'wins' ] += 1
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == first_move), 'total' ] += 1
                elif dealer_result > player_result:
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == first_move), 'losses' ] += 1
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == first_move), 'total' ] += 1
                else:
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == first_move), 'wins' ] += 1
                    results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))) & (results_df['move'] == first_move), 'total' ] += 1

    return results_df
